# Image Color Compression — 02 Preprocessing

There is no missing-value cleaning for an image. “Preprocessing” here means turning the `(H, W, 3)` image into the **pixel matrix** K-Means expects: one row per pixel, three columns (R, G, B), scaled to `[0, 1]`. We also verify the reshape is loss-less (it round-trips exactly).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import utils

china = utils.load_image('data/china.png')
print('image shape:', china.shape)

image shape: (427, 640, 3)


## 1. Reshape image → pixel matrix

`(H, W, 3)` → `(H*W, 3)`, then divide by 255 so every value is in `[0, 1]`.

In [2]:
X, shape = utils.image_to_pixels(china)
print('pixel matrix:', X.shape)
print('value range :', X.min(), '-', X.max())
print('first 3 pixels:\n', np.round(X[:3], 3))

pixel matrix: (273280, 3)
value range : 0.0 - 1.0
first 3 pixels:
 [[0.682 0.788 0.906]
 [0.682 0.788 0.906]
 [0.682 0.788 0.906]]


## 2. Pixel sampling for fast fitting

Fitting K-Means on all 273,280 pixels every iteration is slow. We fit on a random **10,000-pixel sample** and then assign all pixels to the learned centroids — a standard speed-up that barely changes the result because the colour distribution is dense.

In [3]:
rng = np.random.RandomState(42)
idx = rng.choice(len(X), 10000, replace=False)
X_fit = X[idx]
print('sample for fitting:', X_fit.shape)
print('full set to assign:', X.shape)

sample for fitting: (10000, 3)
full set to assign: (273280, 3)


## 3. Round-trip check (loss-less reshape)

Reshaping back to an image and comparing to the original must give **MSE = 0** — proving the only information loss in this project comes from K-Means, not from the preprocessing.

In [4]:
restored = utils.pixels_to_image(X, shape)
print('restored shape:', restored.shape)
print('MSE vs original:', utils.mse(china, restored))
print('identical:', np.array_equal(china, restored))

restored shape: (427, 640, 3)
MSE vs original: 0.0
identical: True


## 4. Note on saved artifacts

Unlike the tabular projects, there is **no `*_cleaned.csv`** to save — the pixel matrix is derived on the fly from the PNG in `data/`. The source images (`china.png`, `flower.png`) are the only stored inputs. Notebook **03** performs the K-Means compression and measures quality vs compression trade-off.